# Inflation et budgets des ménages marocains

Projet académique individuel, première année à l’INSEA. Cette étape décrit les budgets publiés et prépare une simulation de l’exposition aux variations de prix par profil.

## 1. Sources et inventaire du classeur

- **ENNVM 2022–2023.** La collecte du HCP s’étend du 15 mars 2022 au 14 mars 2023. Les tableaux présentent des **dépenses annuelles moyennes par personne** (DAMP, en dirhams courants par personne et par an) et des **coefficients budgétaires** (CB, en pourcentage). La nomenclature est la COICOP Maroc 2022. Les seuils de classes sont intitulés « en 2022 » dans le fichier. [Rapport de l’enquête](https://www.hcp.ma/Rapport-des-resultats-de-l-Enquete-Nationale-sur-le-Niveau-de-Vie-des-Menages-2022-2023_a4062.html)
- **IPC mensuel.** Les divisions de prix en base 100 en 2017 sont tirées de `data/processed/ipc_maroc_mensuel.csv`, de janvier 2017 à octobre 2026. Le HCP décrit une collecte dans 18 villes et un champ urbain. [Note méthodologique sur l’IPC](https://m.hcp.ma/Note-methodologique-sur-l-Indice-des-Prix-a-la-Consommation-Base-100-2017_a4095.html)

| Feuilles inspectées | Contenu et usage |
| --- | --- |
| Page de garde, Table des matières | Identification de l’enquête et repérage des tableaux |
| Considérations importantes | Nomenclature et seuils nationaux, urbains et ruraux des classes de dépense |
| Classification COICOP Maroc 202, Nomenclature B&S | Classification utilisée pour la structure et nomenclature analytique historique |
| Partie1-niveau de la DAMP | Niveau des dépenses, dont le tableau Niv 1 par quintile et caractéristiques du chef du ménage |
| Partie2-Evolution de la DAMP | Comparaisons historiques, dont EV 1 par milieu sur 2014 et 2022 |
| Partie3-Structure de la DAMP | SD 01 par milieu et SD 12 par milieu et quintile. SD 02 porte sur les régions, SD 03 à SD 11 sur d’autres caractéristiques et SD 13 sur les déciles |
| Partie4-Inégalités | Inég 1 mesure la concentration de la masse des dépenses par quintile et milieu |
| Partie 5-DAMPG01 à Partie 5-DAMPG14 | Détail des 14 groupes et de leurs sous-postes, utilisé pour examiner les périmètres |
| Feuil1 | Feuille vide |

**Extraction.** Dans la feuille `Partie3-Structure de la DAMP`, SD 01 est aux lignes 4–22. SD 12 commence à la ligne 288, avec des blocs de postes et totaux aux lignes 294–308 (national), 315–329 (urbain) et 336–350 (rural). Les trois ensembles de SD 12 reprennent les mêmes CB que SD 01 et publient la DAMP avec davantage de décimales. Les seuils sont repris des lignes 31–36 de `Considérations importantes`. Le script `src/clean_households.py` produit les tables chargées ci-dessous.

## 2. Profils et validation des parts

On dispose de 18 profils agrégés : trois ensembles (national, urbain, rural) et cinq quintiles de **dépense annuelle par personne** dans chacun de ces périmètres. Chaque quintile regroupe un cinquième des personnes dans son propre périmètre. Un C1 urbain et un C1 rural occupent la même position relative et reposent sur des seuils en DH différents. Les profils représentent des structures annuelles de dépense. Ils ne contiennent ni trajectoires individuelles, ni observations mensuelles, ni classes de revenu.

Le script additionne les 14 CB publiés et compare la somme au total de 100 %. À un chiffre après la virgule, l’écart théorique maximal sous l’hypothèse d’un arrondi au plus proche est de 14 × 0,05 = 0,70 point de pourcentage. Les coefficients sources sont conservés.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

racine = Path.cwd()
if racine.name == "notebooks":
    racine = racine.parent
traite = racine / "data" / "processed"

# Une ligne de dépense correspond à un groupe pour un profil agrégé.
budgets = pd.read_csv(traite / "depenses_menages_2022_2023.csv")
profils = pd.read_csv(traite / "profils_budgetaires_2022_2023.csv", keep_default_na=False)
correspondance = pd.read_csv(traite / "correspondance_depenses_ipc.csv")
ipc = pd.read_csv(traite / "ipc_maroc_mensuel.csv", parse_dates=["date"], index_col="date")

parts_alimentaires = budgets.loc[budgets["code_groupe"] == 1, [
    "profil", "coefficient_budgetaire_pct"
]].rename(columns={"coefficient_budgetaire_pct": "part_aliments_sans_alcool_pct"})
resume = profils.merge(parts_alimentaires, on="profil", validate="one_to_one")
display(resume[[
    "milieu", "classe_depense", "seuil_2022_libelle_hcp",
    "damp_totale_dh_courants_par_personne_an", "part_aliments_sans_alcool_pct",
    "somme_cb_postes_pct",
]])

,milieu,classe_depense,seuil_2022_libelle_hcp,damp_totale_dh_courants_par_personne_an,part_aliments_sans_alcool_pct,somme_cb_postes_pct
0,National,C1,"C1 : moins de 9148,00 DH",6943.10,46.0,100.0
1,National,C2,"C2 : de 9148,01 DH à moins de 12655,97 DH",10848.12,42.6,100.1
2,National,C3,"C3 :de 12655,98 DH à moins de 17268,00 DH",14792.97,39.6,99.9
3,National,C4,"C4 :de 17268,01 DH à moins de 26255,29 DH",21071.63,36.0,100.1
4,National,C5,"C5 : 26255,30 DH et plus",49633.55,26.7,100.1
5,National,Total,,20658.19,33.4,100.1
6,Urbain,C1,"C1 : moins de 11070,11 DH",8599.46,39.1,100.0
7,Urbain,C2,"C2 : de 11070,12 DH à moins de 15229,13 DH",13171.00,36.9,100.0
8,Urbain,C3,"C3 :de 15229,14 DH à moins de 20815,00 DH",17835.66,35.2,100.0
9,Urbain,C4,"C4 :de 20815,01 DH à moins de 31731,60 DH",25414.87,32.6,99.9


## 3. Audit de la correspondance entre dépenses et divisions IPC

Le rapprochement compare les sous-groupes de l’ENNVM (feuilles DAMPG01 à DAMPG14 et nomenclature COICOP Maroc 2022) aux classes et groupes IPC publiés par le HCP, notamment dans l’annexe du rapport annuel 2024 d’Agadir. Cette annexe éclaire la nomenclature. Les séries de prix préparées pour le calcul restent celles du CSV IPC. Une concordance au niveau de la division ne garantit ni l’identité des articles, ni celle des poids internes. [Détail des classes IPC, HCP](https://www.hcp.ma/region-agadir/docs/docs/rapport%20annuel%202024%20base%202017%20FR.pdf)

- **Proches et retenues dans le scénario prudent** — 01, 03, 05 et 10. Les sous-groupes documentés correspondent à la même fonction de consommation.
- **Partielles, réservées au scénario élargi** — 02, 04, 06, 07, 08, 09, 11 et 13. La division IPC entière sert d’indicateur approché pour le groupe ENNVM. Ce scénario teste la sensibilité à ces choix de périmètre.
- **Absentes** — 12 et 14. Le groupe 12 reste hors des deux scénarios. Le détail annuel IPC identifie notamment les assurances transports et l’intermédiation financière dans « Biens et services divers ». Il ne fournit pas de division mensuelle autonome pour l’ensemble du groupe 12. Réutiliser « Biens et services divers » pour les groupes 12 et 13 ferait porter le même indice composite sur deux postes distincts.
- **Traitement distinct du groupe 14** — Les impôts, transferts et remboursements ne sont pas nécessairement des achats de biens et services. Le CB publié reste visible dans le budget et dans le tableau de couverture, séparément des assurances et services financiers. Le groupe 14 demeure dans le budget total de référence de l’enquête et hors des deux sous-paniers.

In [2]:
# Afficher l’audit dans l’ordre des groupes, avec les divisions absentes laissées vides.
display(correspondance.fillna({"colonne_ipc_candidate": ""}))

,code_groupe,groupe_enquete,colonne_ipc_candidate,statut,justification
0,1,1-Aliments et boissons non alcoolisées,Produits alimentaires et boissons non alcoolisées,proche,DAMPG01 et IPC 011–012 : aliments (dont céréal...
1,2,"2-Boissons alcoolisées, tabac et stupéfiants","Boissons alcoolisées, tabac et stupéfiants",partiel,"DAMPG02 publie une dépense de drogue (23), dis..."
2,3,3-Habillement et chaussures,Articles d'habillement et chaussures,proche,"DAMPG03 et IPC 031–032 : vêtements, chaussures..."
3,4,"4-Logement, eau, électricité, gaz et autres co...","Logement, eau, gaz, électricité et autres comb...",partiel,DAMPG04 comprend les loyers imputés (42). L’IP...
4,5,"5-Ameublement, équipement ménager et entretien...","Meubles, articles de ménage et entretien coura...",proche,"DAMPG05 et IPC 051–056 : ameublement, textiles..."
5,6,6-Santé,Santé,partiel,"Médicaments, soins ambulatoires et hospitalier..."
6,7,7-Transport,Transports,partiel,"Véhicules, utilisation et transport de voyageu..."
7,8,8-Information et communication,Communications,partiel,Recoupement sur la téléphonie. DAMPG08 compren...
8,9,"9-Loisirs, sport et culture",Loisirs et culture,partiel,"Le sport, la culture et la papeterie se recoup..."
9,10,10-Enseignement,Enseignement,proche,DAMPG10 et IPC 101–105 : préscolaire et primai...


## 4. Couverture budgétaire des 18 profils

Chaque couverture est la somme des **coefficients budgétaires d’origine** des groupes retenus, exprimée en pourcentage du budget total ENNVM tel que publié. Les différences d’arrondi du total observé (99,8 % à 100,1 %) sont conservées. La couverture élargie inclut les rapprochements partiels. Elle mesure la part du budget à laquelle on attribuerait une série de prix dans ce scénario.

| Scénario | Groupes ENNVM retenus | Interprétation |
| --- | --- | --- |
| Prudent | 01, 03, 05, 10 | Sous-panier des correspondances proches validées au niveau des divisions |
| Élargi | 01 à 11 et 13 | Même sous-panier, complété par les correspondances partielles pour tester la sensibilité |

Le tableau ci-dessous porte uniquement sur les budgets. Aucun taux d’inflation par profil n’est calculé.

In [3]:
# Lire les groupes retenus depuis le même tableau que celui de l’audit.
if sorted(correspondance["code_groupe"].tolist()) != list(range(1, 15)):
    raise ValueError("La correspondance doit contenir exactement les 14 groupes ENNVM.")
if not correspondance["statut"].isin(["proche", "partiel", "absent"]).all():
    raise ValueError("Statut de correspondance inconnu.")

codes_proches = correspondance.loc[correspondance["statut"] == "proche", "code_groupe"].tolist()
codes_partiels = correspondance.loc[correspondance["statut"] == "partiel", "code_groupe"].tolist()
scenarios = {
    "prudent": codes_proches,
    "elargi": sorted(codes_proches + codes_partiels),
}
mois_reference_travail = "2022-09"


def coefficients_budgetaires():
    """Vérifier les CB puis les organiser par profil et groupe."""
    if profils["profil"].duplicated().any() or profils["profil"].empty:
        raise ValueError("La table des profils doit contenir des identifiants uniques.")
    if set(budgets["profil"]) != set(profils["profil"]):
        raise ValueError("Les profils des budgets et du récapitulatif diffèrent.")
    if set(budgets["code_groupe"]) != set(range(1, 15)):
        raise ValueError("Un groupe ENNVM est absent ou inattendu dans les budgets.")
    if budgets.duplicated(["profil", "code_groupe"]).any():
        raise ValueError("Un groupe de dépenses apparaît plusieurs fois pour un profil.")

    cb = budgets.pivot(index="profil", columns="code_groupe", values="coefficient_budgetaire_pct")
    cb = cb.reindex(profils["profil"]).apply(pd.to_numeric, errors="raise")
    if not np.isfinite(cb.to_numpy(dtype=float, na_value=np.nan)).all() or cb.lt(0).any().any():
        raise ValueError("Les CB doivent être renseignés, finis et positifs ou nuls.")
    if cb.sum(axis=1).sub(100).abs().gt(14 * 0.05 + 1e-9).any():
        raise ValueError("Le total des CB d’un profil sort de la tolérance d’arrondi.")
    return cb


cb = coefficients_budgetaires()
couverture = profils.set_index("profil")[["milieu", "classe_depense", "somme_cb_postes_pct"]].copy()
couverture["Prudent (%)"] = cb[scenarios["prudent"]].sum(axis=1).round(1)
couverture["Élargi (%)"] = cb[scenarios["elargi"]].sum(axis=1).round(1)
couverture["Groupe 12 (%)"] = cb[12]
couverture["Groupe 14 (%)"] = cb[14]
display(couverture)

,milieu,classe_depense,somme_cb_postes_pct,Prudent (%),Élargi (%),Groupe 12 (%),Groupe 14 (%)
profil,,,,,,,
National_C1,National,C1,100.0,51.9,98.0,1.1,0.9
National_C2,National,C2,100.1,48.8,97.1,1.8,1.2
National_C3,National,C3,99.9,46.2,95.9,2.6,1.4
National_C4,National,C4,100.1,43.0,94.5,3.7,1.9
National_C5,National,C5,100.1,36.1,90.1,5.1,4.9
National_Total,National,Total,100.1,41.4,93.2,3.8,3.1
Urbain_C1,Urbain,C1,100.0,45.0,97.7,1.5,0.8
Urbain_C2,Urbain,C2,100.0,43.5,96.2,2.7,1.1
Urbain_C3,Urbain,C3,100.0,42.3,95.1,3.5,1.4


## 5. Calcul préparé pour un panier fixe

Le mois de référence de travail proposé est **septembre 2022**, proche du milieu de la collecte du 15 mars 2022 au 14 mars 2023. Ce repère fixe l’indice simulé à 100 en septembre 2022. Les poids décrivent l’ensemble de l’enquête annuelle. Ils ne représentent pas les achats effectués ce mois-là, et un autre mois de référence peut modifier les taux obtenus à poids fixes. [Rapport ENNVM 2022–2023 du HCP](https://www.hcp.ma/Rapport-des-resultats-de-l-Enquete-Nationale-sur-le-Niveau-de-Vie-des-Menages-2022-2023_a4062.html)

Pour un scénario de groupes $S$, la couverture du profil $p$ vaut $C_p=\sum_{k\in S} CB_{p,k}$. Dans le calcul du sous-panier uniquement, on divise chaque coefficient retenu par $C_p$ pour que ces poids totalisent 1. Cette renormalisation se fait en mémoire et les coefficients publiés restent inchangés.

$$
w_{p,k} = CB_{p,k}/C_p,\qquad
P_{p,t}=100\sum_{k\in S} w_{p,k}\frac{I_{k,t}}{I_{k,b}}.
$$

Ici $I_{k,t}$ désigne l’IPC de la division associée au mois $t$, et $b$ le mois de référence commun. Le taux du panier est $100(P_{p,t}/P_{p,t-h}-1)$, avec $h=1$ pour la variation mensuelle et $h=12$ pour le glissement annuel. Le premier mois et les douze premiers mois, respectivement, restent sans taux.

La fonction exige un scénario explicite (`prudent` ou `elargi`). Elle est définie ci-dessous et son calcul reste en attente.

In [4]:
def preparer_panier(scenario, mois_reference):
    """Vérifier les séries, les coefficients et la date avant tout calcul."""
    if not isinstance(scenario, str) or scenario not in scenarios:
        raise ValueError("Choisir explicitement 'prudent' ou 'elargi'.")
    codes = scenarios[scenario]
    if not codes or len(codes) != len(set(codes)):
        raise ValueError("Le scénario doit contenir des groupes distincts.")
    if set(codes) & {12, 14}:
        raise ValueError("Les groupes 12 et 14 sont exclus des deux scénarios.")

    liens = correspondance.set_index("code_groupe").reindex(codes)
    statuts_autorises = ["proche"] if scenario == "prudent" else ["proche", "partiel"]
    if not liens["statut"].isin(statuts_autorises).all():
        raise ValueError("Un groupe est absent de la correspondance ou interdit dans ce scénario.")
    colonnes = liens["colonne_ipc_candidate"]
    if colonnes.isna().any() or colonnes.duplicated().any() or ipc.columns.has_duplicates:
        raise ValueError("Les divisions IPC associées doivent être renseignées et uniques.")
    absentes = colonnes[~colonnes.isin(ipc.columns)].tolist()
    if absentes:
        raise ValueError(f"Divisions absentes du CSV IPC : {absentes}.")

    # Le décalage de 1 ou 12 lignes n’est valable qu’avec un calendrier complet.
    indices = ipc[colonnes.tolist()].sort_index().apply(pd.to_numeric, errors="raise")
    if indices.empty or not isinstance(indices.index, pd.DatetimeIndex):
        raise ValueError("Le CSV IPC doit contenir des dates mensuelles.")
    if indices.index.hasnans or indices.index.has_duplicates or indices.index.tz is not None:
        raise ValueError("Dates IPC manquantes, en double ou avec fuseau horaire.")
    calendrier = pd.date_range(indices.index.min().normalize(), indices.index.max().normalize(), freq="MS")
    if not indices.index.equals(calendrier):
        raise ValueError("Le calendrier IPC doit être complet et daté au premier de chaque mois.")
    if not np.isfinite(indices.to_numpy(dtype=float, na_value=np.nan)).all() or indices.le(0).any().any():
        raise ValueError("Les IPC retenus doivent être renseignés, finis et strictement positifs.")

    if not isinstance(mois_reference, str) or not re.fullmatch(r"[0-9]{4}-[0-9]{2}(-01)?", mois_reference):
        raise ValueError("Saisir le mois de référence au format AAAA-MM ou AAAA-MM-01.")
    date_iso = mois_reference + "-01" if len(mois_reference) == 7 else mois_reference
    reference = pd.to_datetime(date_iso, format="%Y-%m-%d", errors="coerce")
    if pd.isna(reference) or reference not in indices.index:
        raise ValueError("Le mois de référence doit être un mois valide présent dans le CSV IPC.")

    indices.columns = codes
    coefficients = coefficients_budgetaires()[codes]
    couverture = coefficients.sum(axis=1).rename("somme_cb_retenus_pct")
    if couverture.le(0).any():
        raise ValueError("Les coefficients retenus doivent avoir une somme positive pour chaque profil.")
    return indices, reference, coefficients, couverture


def simuler_panier(scenario, mois_reference=mois_reference_travail):
    """Calculer les taux du sous-panier pour le scénario explicitement choisi."""
    indices, reference, coefficients, couverture = preparer_panier(scenario, mois_reference)

    # Le budget complet et ses coefficients publiés restent inchangés.
    poids = coefficients.div(couverture, axis=0)
    ratios = indices.div(indices.loc[reference])
    cout_relatif = 100 * ratios.dot(poids.T)
    return {
        "couverture": couverture,
        "indice_panier": cout_relatif,
        "inflation_mensuelle_pct": 100 * (cout_relatif / cout_relatif.shift(1) - 1),
        "glissement_annuel_pct": 100 * (cout_relatif / cout_relatif.shift(12) - 1),
    }

## 6. Limites scientifiques

- **Collecte urbaine.** Le HCP collecte les prix dans 18 villes. Pour les profils ruraux, ces séries peuvent servir de scénario commun de prix. Les écarts simulés entre profils refléteraient les différences de poids sous les mêmes prix, et ne mesureraient pas les écarts de prix entre marchés urbains et ruraux. Les lieux d’achat, l’accès aux produits, les prix locaux et l’autoconsommation peuvent limiter la représentativité du scénario. [Méthodologie de l’IPC du HCP](https://m.hcp.ma/Note-methodologique-sur-l-Indice-des-Prix-a-la-Consommation-Base-100-2017_a4095.html)
- **Poids fixes et champ du panier.** Le scénario prudent ne couvre qu’une fraction du budget. Le scénario élargi gagne en couverture en utilisant des indices approchés. Comparer les deux mesure la sensibilité conjointe au périmètre et aux rapprochements. Appliquer les poids 2022–2023 à d’autres années constitue une simulation, avec une composition du panier tenue fixe.
- **Niveau de vie et pouvoir d’achat.** Les quintiles sont définis sur les dépenses annuelles par personne, avec des seuils propres à chaque milieu. Les données agrégées ne permettent pas d’observer les substitutions individuelles. Une perte effective de pouvoir d’achat demande des données de revenu appropriées.